# SUES-200 exploration

https://github.com/Yux1angJi/SUES-200

```txt
=== SUES-200 Dataset ===

Dataset structure:
- Drone images with GPS coordinates
- Satellite reference images
- Different format from UAV-VisLoc
```


In [1]:
import os
from pathlib import Path

import cv2
import fiftyone as fo
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from PIL import Image
from rasterio.enums import Resampling

DATA = Path("data/sues200")

/Users/yugisu/Personal/visual-geolocalization/diffusion-vpr/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Get the SUES-200 dataset
![ -f "{DATA}.zip" ] || gdown "1UyVyFJ_pRaJHIr_eBY2HL7gkS5y9UxqI" -O "{DATA}.zip"
![ -d "{DATA}" ] || unzip -q "{DATA}.zip" -d "{DATA.parent}"
# SUES-200 might extract to a different folder name, adjust if needed
![ -d "{DATA}" ] || mv "{DATA.parent}/SUES-200" "{DATA}"

Downloading...
From (original): https://drive.google.com/uc?id=1UyVyFJ_pRaJHIr_eBY2HL7gkS5y9UxqI
From (redirected): https://drive.google.com/uc?id=1UyVyFJ_pRaJHIr_eBY2HL7gkS5y9UxqI&confirm=t&uuid=474ad7f3-eb89-424e-968c-83932829bfa8
To: /Users/yugisu/Personal/visual-geolocalization/diffusion-vpr/data/sues200.zip
  1%|▎                                     | 53.0M/5.70G [00:02<04:43, 19.9MB/s]^C
object address  : 0x10bad9840
object refcount : 3
object type     : 0x1074a9770
object type name: KeyboardInterrupt
object repr     : KeyboardInterrupt()
lost sys.stderr
  1%|▎                                     | 55.6M/5.70G [00:03<05:45, 16.3MB/s]
unzip:  cannot find or open data/sues200.zip, data/sues200.zip.zip or data/sues200.zip.ZIP.
mv: rename data/SUES-200 to data/sues200: No such file or directory


In [ ]:
flight_id = "03"
csv_path = DATA / flight_id / f"{flight_id}.csv"
drone_dir = DATA / flight_id / "drone"

df = pd.read_csv(csv_path)

uav_dataset = fo.Dataset()

samples = []
for _, row in df.iterrows():
  image_path = os.path.join(drone_dir, row["filename"])

  if os.path.exists(image_path):
    sample = fo.Sample(
      filepath=image_path,
      type="UAV",
      lat=row["lat"],
      lon=row["lon"],
      location=fo.GeoLocation(point=(row["lon"], row["lat"])),
      # UAV metadata
      altitude=row["height"],
      omega=row["Omega"],  # pitch
      kappa=row["Kappa"],  # roll
      phi1=row["Phi1"],  # yaw (higher confidence)
      phi2=row["Phi2"],  # yaw (lower confidence)
      date=row["date"],
    )
    samples.append(sample)

uav_dataset.add_samples(samples);

In [ ]:
def visualize_flight_trajectory(data_path, flight_id, scale_factor=0.1):
  """Load satellite image and visualize drone flight trajectory."""

  # Load satellite image
  sat_path = data_path / flight_id / f"satellite{flight_id}.tif"
  with rasterio.open(sat_path) as src:
    h, w = int(src.height * scale_factor), int(src.width * scale_factor)
    data = src.read(out_shape=(src.count, h, w), resampling=Resampling.lanczos)
    sat_img = (
      cv2.cvtColor(np.transpose(data[:3], (1, 2, 0)).astype(np.uint8), cv2.COLOR_RGB2BGR)
      if src.count >= 3
      else np.transpose(data[0], (1, 2, 0)).astype(np.uint8)
    )

  # Load satellite bounds
  sat_df = pd.read_csv(data_path / "satellite_coordinates_range.csv")
  sat_df.columns = ["mapname", "LT_lat", "LT_lon", "RB_lat", "RB_lon", "region"]
  bounds = sat_df[sat_df["mapname"] == f"{flight_id}.tif"].iloc[0]

  # Convert GPS to pixel coordinates
  def gps_to_pixel(lat, lon):
    x = (lon - bounds["LT_lon"]) / (bounds["RB_lon"] - bounds["LT_lon"]) * w
    y = (bounds["LT_lat"] - lat) / (bounds["LT_lat"] - bounds["RB_lat"]) * h
    return int(x), int(y)

  # Draw trajectory
  drone_df = df.sort_values("num")
  coords = [gps_to_pixel(row["lat"], row["lon"]) for _, row in drone_df.iterrows()]

  result = sat_img.copy()
  for i in range(len(coords) - 1):
    cv2.line(result, coords[i], coords[i + 1], (0, 255, 0), 2)

  for coord in coords:
    cv2.circle(result, coord, 3, (0, 255, 0), 4)
  cv2.circle(result, coords[0], 8, (255, 0, 0), 6)
  cv2.circle(result, coords[-1], 8, (0, 0, 255), 6)

  # Display
  print(f"Total waypoints: {len(coords)} | Image size: {w}x{h}")

  plt.figure(figsize=(14, 10))
  plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
  plt.title(f"UAV Flight Trajectory - Flight {flight_id}")
  plt.xlabel("X (pixels)")
  plt.ylabel("Y (pixels)")
  plt.tight_layout()
  plt.show()


# Run visualization
visualize_flight_trajectory(DATA, flight_id)


In [ ]:
def load_satellite_image(data_path, flight_id, scale_factor=0.1):
  """Load and downscale satellite image with metadata.

  Args:
    data_path: Path to dataset root
    flight_id: Flight identifier (e.g., "03")
    scale_factor: Downscale factor (0.1 = 10% of original)

  Returns:
    tuple: (sat_img, bounds, dimensions)
      - sat_img: numpy array (H, W, C)
      - bounds: bbox tuple (lat_min, lon_min, lat_max, lon_max)
      - dimensions: dict with height, width, scale_factor
  """
  sat_path = data_path / flight_id / f"satellite{flight_id}.tif"

  # Load and downscale image
  with rasterio.open(sat_path) as src:
    h, w = int(src.height * scale_factor), int(src.width * scale_factor)
    data = src.read(out_shape=(src.count, h, w), resampling=Resampling.lanczos)
    sat_img = (
      np.transpose(data[:3], (1, 2, 0)).astype(np.uint8)
      if src.count >= 3
      else np.transpose(data[0], (1, 2, 0)).astype(np.uint8)
    )

  # Load coordinate bounds
  sat_df = pd.read_csv(data_path / "satellite_coordinates_range.csv")
  sat_df.columns = ["mapname", "LT_lat", "LT_lon", "RB_lat", "RB_lon", "region"]
  row = sat_df[sat_df["mapname"] == f"{flight_id}.tif"].iloc[0]

  # bbox format: (lat_min, lon_min, lat_max, lon_max)
  bounds = (row["RB_lat"], row["LT_lon"], row["LT_lat"], row["RB_lon"])
  dimensions = {"height": h, "width": w, "scale_factor": scale_factor}

  return sat_img, bounds, dimensions


def create_chunks(sat_img, bounds, dimensions, chunk_size=256, stride=None):
  """Split satellite image into chunks with GPS coordinates.

  Args:
    sat_img: Satellite image array (H, W, C)
    bounds: bbox tuple (lat_min, lon_min, lat_max, lon_max)
    dimensions: Dict with height, width, scale_factor
    chunk_size: Size of each tile in pixels
    stride: Step size between tiles (if None, equals chunk_size)

  Returns:
    List of chunk dicts with rgb, coordinates, and metadata
  """
  stride = stride or chunk_size
  h, w = dimensions["height"], dimensions["width"]
  lat_min, lon_min, lat_max, lon_max = bounds

  def pixel_to_gps(x, y):
    lat = lat_max - (y / h) * (lat_max - lat_min)
    lon = lon_min + (x / w) * (lon_max - lon_min)
    return lat, lon

  chunks = []
  for y in range(0, h - chunk_size, stride):
    for x in range(0, w - chunk_size, stride):
      chunk = sat_img[y : y + chunk_size, x : x + chunk_size]
      cx, cy = x + chunk_size // 2, y + chunk_size // 2
      lat, lon = pixel_to_gps(cx, cy)

      # Chunk bbox: (lat_min, lon_min, lat_max, lon_max)
      chunk_lat_min, chunk_lon_min = pixel_to_gps(x, y + chunk_size)
      chunk_lat_max, chunk_lon_max = pixel_to_gps(x + chunk_size, y)

      chunks.append(
        {
          "rgb": chunk,
          "x": x,
          "y": y,
          "lat": lat,
          "lon": lon,
          "bbox_pixels": (x, y, x + chunk_size, y + chunk_size),
          "bbox_gps": (chunk_lat_min, chunk_lon_min, chunk_lat_max, chunk_lon_max),
          "scale_factor": dimensions["scale_factor"],
          "resolution": 0.28 / dimensions["scale_factor"],
          "chunk_size": chunk_size,
          "chunk_stride": stride,
        }
      )

  print(f"Created {len(chunks)} chunks from {w}x{h} image (scale={dimensions['scale_factor']})")
  return chunks


def visualize_chunks(chunks, sat_img):
  """Visualize chunk grid on satellite image with statistics."""

  print(f"\nChunk Statistics:")
  print(f"  Total chunks: {len(chunks)}")
  print(f"  Chunk size: {chunks[0]['chunk_size']}x{chunks[0]['chunk_size']} pixels")
  print(f"  Stride: {chunks[0]['chunk_stride']} pixels")
  print(f"  Resolution: {chunks[0]['resolution']:.2f} m/pixel")
  print(f"  First chunk center: ({chunks[0]['lat']:.6f}, {chunks[0]['lon']:.6f})")
  print(f"  First chunk GPS bbox: {chunks[0]['bbox_gps']}")

  fig, ax = plt.subplots(figsize=(12, 10))
  ax.imshow(cv2.cvtColor(sat_img, cv2.COLOR_BGR2RGB))

  for chunk in chunks[::10]:
    x1, y1, x2, y2 = chunk["bbox_pixels"]
    rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=1, edgecolor="r", facecolor="none")
    ax.add_patch(rect)

  ax.set_title(f"Satellite Image Chunks (showing 1 in 10)")
  plt.tight_layout()
  plt.show()


# Load and chunk satellite image
sat_img, bounds, dims = load_satellite_image(DATA, flight_id, scale_factor=0.2)
chunks = create_chunks(sat_img, bounds, dims, chunk_size=256, stride=128)

# Visualize
visualize_chunks(chunks, sat_img)


In [ ]:
root = Path("data/visloc-post-satellite")

satellite_dataset = fo.Dataset()

for sf in [0.1, 0.2, 0.3]:
  sat_img, bounds, dims = load_satellite_image(DATA, flight_id, scale_factor=sf)
  chunks = create_chunks(sat_img, bounds, dims, chunk_size=256, stride=128)

  samples = []
  for idx, chunk in enumerate(chunks):
    img = Image.fromarray(chunk["rgb"])
    chunk_path = root / f"scale-{sf}" / f"chunk_{idx:05d}.png"

    chunk_path.parent.mkdir(parents=True, exist_ok=True)
    img.save(chunk_path)

    sample = fo.Sample(
      filepath=chunk_path,
      type="satellite",
      lat=chunk["lat"],
      lon=chunk["lon"],
      location=fo.GeoLocation(point=(chunk["lon"], chunk["lat"])),
      # Satellite metadata
      bbox_pixels=chunk["bbox_pixels"],
      bbox_gps=chunk["bbox_gps"],
      scale_factor=chunk["scale_factor"],
      resolution=chunk["resolution"],
      chunk_size=chunk["chunk_size"],
      chunk_stride=chunk["chunk_stride"],
    )
    samples.append(sample)

  satellite_dataset.add_samples(samples)

In [ ]:
session = fo.launch_app(satellite_dataset)
